# G1 Academy Bonus - Task 4: Vision -- "What Do You See?"


## Introduction
Extend Task 3's assistant so it can answer questions about what its camera sees, e.g. "What do you see?" or "Describe what's in front of you" (also works in German: "Beschreibe, was du siehst").

- `g1.get_rgbd()` returns a dict; `["rgb_jpeg"]` is the current camera frame as raw JPEG bytes (or the whole dict is `None` if no camera stream is available -- check before using it).
- `ollama_api_util.describe_image(client, jpeg_bytes, question)` sends one frame plus a text question to a vision-capable model and returns its text answer.

This notebook sets up its own `g1`/`client`/`history`/`knowledge`/`chat_turn_with_knowledge`/`GESTURES`/`MOVES`/`chat_turn_with_actions` (you do not need to have run Tasks 1-3 first).


In [ ]:
import sys
sys.path.append("..")
from sdk_wrapper import G1
import ollama_api_util

g1 = G1(iface="eth0", domain_id=0)
client = ollama_api_util.get_client()
history = []

knowledge = ollama_api_util.load_knowledge_base("g1_academy_knowledge_de.json")

def chat_turn_with_knowledge(text):
    matches = ollama_api_util.retrieve_relevant(knowledge, text, top_k=3)
    if matches:
        context = "\n".join(f"F: {m['question']}\nA: {m['answer']}" for m in matches)
        augmented = f"Known facts:\n{context}\n\nQuestion: {text}"
    else:
        augmented = text  # nothing relevant found -- ask it plainly, don't force in noise
    reply = ollama_api_util.chat_reply(client, augmented, history=history)
    history.append({"role": "user", "content": text})
    history.append({"role": "assistant", "content": reply})
    print(reply)
    g1.say(reply, language="EN")
    return reply

GESTURES = {
    "wave": g1.face_wave, "high wave": g1.high_wave, "clap": g1.clap,
    "shake hand": g1.shake_hand, "hug": g1.hug, "kiss": g1.left_kiss,
}
MOVES = {
    "forward": (0.15, 0, 0), "backward": (-0.15, 0, 0),
    "left": (0, 0.15, 0), "right": (0, -0.15, 0),
    "turn left": (0, 0, 0.5), "turn right": (0, 0, -0.5),
}

def chat_turn_with_actions(text):
    # Classified by keyword match rather than a second LLM call -- more reliable than
    # parsing a small model's free-form classification, and avoids doubling latency.
    # Longest names checked first so "turn left"/"turn right" win over plain "left"/"right".
    lowered = text.lower()
    gesture_name = next((name for name in sorted(GESTURES, key=len, reverse=True) if name in lowered), None)
    if gesture_name:
        reply = f"Okay, {gesture_name} now."
        g1.say(reply, language="EN")
        GESTURES[gesture_name]()
        history.append({"role": "user", "content": text})
        history.append({"role": "assistant", "content": reply})
        print(reply)
        return reply
    move_name = next((name for name in sorted(MOVES, key=len, reverse=True) if name in lowered), None)
    if move_name:
        reply = f"Okay, moving {move_name} now."
        g1.say(reply, language="EN")
        g1.loco_move(*MOVES[move_name], duration_s=1.0)
        history.append({"role": "user", "content": text})
        history.append({"role": "assistant", "content": reply})
        print(reply)
        return reply
    return chat_turn_with_knowledge(text)


In [ ]:
# Run this only after the area is clear, before issuing movement commands.
g1.walk_mode(waist_locked=False)


In [ ]:
frame = g1.get_rgbd()
assert frame and frame.get("rgb_jpeg"), "No camera frame available -- is the camera stream running?"


## Codex prompt
```
I already have chat_turn_with_actions(text) from Task 3. I also have g1.get_rgbd() (returns a dict
with a "rgb_jpeg" key holding the current camera frame as JPEG bytes, or None if unavailable), and
ollama_api_util.describe_image(client, jpeg_bytes, question) which answers a question about one frame.

Write a new function chat_turn_with_vision(text) that:
1. Detects whether text is a vision question (e.g. contains "siehst du", "was siehst", "kamera",
   "beschreibe") -- ask the model to classify it if a simple keyword check feels too fragile.
2. If so: calls g1.get_rgbd(), checks the frame is not None, calls
   ollama_api_util.describe_image(client, frame["rgb_jpeg"], text), prints the answer, and speaks it
   with g1.say(answer, language="EN").
3. Otherwise, fall back to chat_turn_with_actions(text).

Handle a None frame from get_rgbd() by speaking a short English error message instead of crashing.
```


In [ ]:
VISION_TRIGGERS = ("see", "camera", "kamera", "describe", "look", "siehst", "beschreib")

def chat_turn_with_vision(text):
    if not any(trigger in text.lower() for trigger in VISION_TRIGGERS):
        return chat_turn_with_actions(text)
    frame = g1.get_rgbd()
    if not frame or not frame.get("rgb_jpeg"):
        reply = "I can't see anything right now -- the camera stream isn't available."
    else:
        reply = ollama_api_util.describe_image(client, frame["rgb_jpeg"], text)
    print(reply)
    g1.say(reply, language="EN")
    history.append({"role": "user", "content": text})
    history.append({"role": "assistant", "content": reply})
    return reply


In [ ]:
chat_turn_with_vision(input("Du: "))
